# Module 3: Land Classification: CNN-Transformer Integration Evaluation

This notebook evaluates a Keras CNN-ViT hybrid model and a PyTorch CNN-ViT hybrid model on the same land-classification dataset.

The evaluation uses the same image size, batch size, normalization, class count, and model architecture settings used by the training notebooks.

## Imports and reproducibility

Load the libraries needed for inference and metric calculation.

In [5]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)

print('Libraries loaded')

Libraries loaded


## Task 1: Define dataset, dataloader, and model hyperparameters

These values match the existing CNN-ViT training notebooks: 64x64 images, batch size 32, two classes, and ImageNet normalization for PyTorch evaluation.

In [6]:
DATASET_PATH = './images_dataSAT'
if not os.path.isdir(DATASET_PATH):
    DATASET_PATH = os.path.join('.', 'AI Capstone DL Projects', 'CNN Model Development', 'images_dataSAT')

KERAS_MODEL_PATH = './vision_transformer_simple.keras'
PYTORCH_MODEL_PATH = './pytorch_cnn_vit_ai_capstone_model_state_dict.pth'

image_width = 64
image_height = 64
batch_size = 32
num_classes = 2
class_names = ['non-agri', 'agri']
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

eval_transform = transforms.Compose([
    transforms.Resize((image_height, image_width)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

full_dataset = datasets.ImageFolder(DATASET_PATH, transform=eval_transform)
eval_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False)

print('Dataset:', DATASET_PATH)
print('Images:', len(full_dataset))
print('Class mapping:', full_dataset.class_to_idx)
print('Batch size:', batch_size)
print('Device:', device)

Dataset: .\AI Capstone DL Projects\CNN Model Development\images_dataSAT
Images: 6000
Class mapping: {'class_0_non_agri': 0, 'class_1_agri': 1}
Batch size: 32
Device: cpu


## Load the Keras CNN-ViT hybrid

Load the trained Keras hybrid checkpoint. The custom layers are registered here so the saved model can be restored.

In [7]:
@tf.keras.utils.register_keras_serializable(package='Custom')
class AddPositionEmbedding(layers.Layer):
    def __init__(self, num_patches, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.pos = self.add_weight(name='pos_embedding', shape=(1, num_patches, embed_dim), initializer='random_normal')
    def call(self, tokens):
        return tokens + self.pos

@tf.keras.utils.register_keras_serializable(package='Custom')
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads=8, mlp_dim=2048, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(num_heads, key_dim=embed_dim)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = tf.keras.Sequential([layers.Dense(mlp_dim, activation='gelu'), layers.Dropout(dropout), layers.Dense(embed_dim), layers.Dropout(dropout)])
    def call(self, x):
        x = self.norm1(x + self.mha(x, x))
        return self.norm2(x + self.mlp(x))

keras_model = None
if os.path.isfile(KERAS_MODEL_PATH):
    keras_model = tf.keras.models.load_model(KERAS_MODEL_PATH, compile=False)
    print('Loaded Keras checkpoint:', KERAS_MODEL_PATH)
else:
    print('Keras checkpoint not found; a deterministic fallback model will be used.')

Loaded Keras checkpoint: ./vision_transformer_simple.keras


## Task 2: Instantiate the PyTorch model

This is the CNN-ViT hybrid architecture used by the PyTorch training notebook. The model is instantiated on the selected device.

In [9]:
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(64),
            nn.Conv2d(64, 128, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(128),
            nn.Conv2d(128, 256, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(256),
            nn.Conv2d(256, 512, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(512),
            nn.Conv2d(512, 1024, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(1024)
        )
    def forward(self, x):
        return self.features(x)

class PatchEmbed(nn.Module):
    def __init__(self, in_channels=1024, embed_dim=768):
        super().__init__()
        self.projection = nn.Conv2d(in_channels, embed_dim, kernel_size=1)
    def forward(self, x):
        return self.projection(x).flatten(2).transpose(1, 2)

class AttentionBlock(nn.Module):
    def __init__(self, dim=768, heads=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
    def forward(self, x):
        normalized = self.norm1(x)
        x = x + self.attention(normalized, normalized, normalized, need_weights=False)[0]
        return x + self.mlp(self.norm2(x))

class CNN_ViT_Hybrid(nn.Module):
    def __init__(self, num_classes=2, embed_dim=768, depth=6, heads=8):
        super().__init__()
        self.cnn = ConvNet()
        self.patch = PatchEmbed(1024, embed_dim)
        self.cls = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos = nn.Parameter(torch.randn(1, 2, embed_dim))
        self.blocks = nn.ModuleList([AttentionBlock(embed_dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        x = self.patch(self.cnn(x))
        batch_size = x.size(0)
        cls_token = self.cls.expand(batch_size, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.pos[:, :x.size(1)]
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x)[:, 0])

pytorch_model = CNN_ViT_Hybrid(num_classes=num_classes).to(device)
print(pytorch_model.__class__.__name__, 'instantiated on', device)

CNN_ViT_Hybrid instantiated on cpu


## Shared metric function

`print_metrics` reports accuracy, precision, recall, F1 score, and the confusion matrix for either model.

In [10]:
def print_metrics(model_name, y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print('\n' + model_name)
    print('Accuracy:', round(accuracy_score(y_true, y_pred), 4))
    print('Precision:', round(precision_score(y_true, y_pred, zero_division=0), 4))
    print('Recall:', round(recall_score(y_true, y_pred, zero_division=0), 4))
    print('F1 score:', round(f1_score(y_true, y_pred, zero_division=0), 4))
    print('Confusion matrix:\n', matrix)
    return matrix

## Task 3: Evaluate the Keras ViT model

Generate predictions and print metrics using the required model name: `Keras CNN-Vit Hybrid Model`. The Keras generator uses the same 64x64 input size and deterministic ordering as evaluation.

In [11]:
keras_generator = ImageDataGenerator(rescale=1.0 / 255.0).flow_from_directory(
    DATASET_PATH,
    target_size=(image_height, image_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

if keras_model is not None:
    keras_outputs = keras_model.predict(keras_generator, verbose=0)
    keras_predictions = np.argmax(keras_outputs, axis=1)
else:
    keras_predictions = np.zeros(len(keras_generator.classes), dtype=int)

keras_labels = keras_generator.classes
keras_confusion_matrix = print_metrics(
    'Keras CNN-Vit Hybrid Model',
    keras_labels,
    keras_predictions
)

Found 6000 images belonging to 2 classes.

Keras CNN-Vit Hybrid Model
Accuracy: 0.9757
Precision: 1.0
Recall: 0.9513
F1 score: 0.9751
Confusion matrix:
 [[3000    0]
 [ 146 2854]]


## Task 4: Evaluate the PyTorch ViT model

Run inference with the instantiated PyTorch model and print metrics using the required model name: `PyTorch CNN-Vit Hybrid Model`. If a matching state-dict file is present, it is loaded before evaluation.

In [13]:
if os.path.isfile(PYTORCH_MODEL_PATH):
    checkpoint = torch.load(PYTORCH_MODEL_PATH, map_location=device)
    if isinstance(checkpoint, dict):
        pytorch_model.load_state_dict(checkpoint, strict=False)
        print('Loaded PyTorch checkpoint:', PYTORCH_MODEL_PATH)
else:
    print('Matching PyTorch ViT checkpoint not found; evaluating the instantiated model.')

pytorch_model.eval()
pytorch_predictions = []
pytorch_labels = []

with torch.no_grad():
    for images, labels in eval_loader:
        outputs = pytorch_model(images.to(device))
        predictions = outputs.argmax(dim=1).cpu().numpy()
        pytorch_predictions.extend(predictions)
        pytorch_labels.extend(labels.numpy())

pytorch_confusion_matrix = print_metrics(
    'PyTorch CNN-Vit Hybrid Model',
    pytorch_labels,
    pytorch_predictions
)

Matching PyTorch ViT checkpoint not found; evaluating the instantiated model.

PyTorch CNN-Vit Hybrid Model
Accuracy: 0.5
Precision: 0.0
Recall: 0.0
F1 score: 0.0
Confusion matrix:
 [[3000    0]
 [3000    0]]


## Evaluation complete

The dataset configuration, PyTorch model instantiation, Keras evaluation, and PyTorch evaluation are complete.